<a href="https://colab.research.google.com/github/Mansi-3s/Creditt_wise_loan/blob/main/CNN_for_CIFAR10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision # pytorch lib, for computer vision on it dataset like: CIFAR10, MNIST..., also has pretrained CNNs, utilities for image transformation

#Steps:

Image(0-255) ->> Scale(0,1) ->> Normalize(-1, +1)

In [3]:
from torchvision.datasets import CIFAR10

In [4]:
#DATASETS & DATALOADERS
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

#image => size(0, 1) => normalize(-1, 1)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

testset = CIFAR10(root="./data", train=False, download=True, transform=transform)
trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)


100%|██████████| 170M/170M [00:13<00:00, 12.9MB/s]


In [5]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [6]:
trainloader = DataLoader(trainset, batch_size=128, shuffle=True)
testloader =  DataLoader(trainset, batch_size=128)

####**CNN Arcitecture**

input image size is  (32 * 32 * 3 )


CNN Layer: input 3 * 32 * 32 -->>>(convolution layer + ReLU [32, 32, 32] > maxPooling [16, 16, 32] ) -> (convolution layer + ReLU [16, 16, 64] > maxPooling [8, 8, 64] )-> (convolution layer + ReLU [8, 8, 128] > maxPooling [4,4,128] ) -->>>  Flattening (4 * 4 * 128)  -->>> Fully connected layer ( Linear+ ReLU >> >> Linear(o/p = 10))... {FC layer is simple ANN, on it we have one hidden layer annd one output layer}

kernel value = 3*3, padding = 1 for convolution layer

kernel value = 2*2, stride_value = 2 for maxPooling
## ques: Multi-class classification (CNN for image_classification)
Loss_function = CrossEntropy (have automatic softmax in output layer) that's why we take last layer as Linear .


we have 10 classes, so output = 10

### Build the CNN

In [7]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN, self).__init__()
    self.conv_layer = nn.Sequential(
        nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(32, 64, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64, 128, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2)
    )

    # output = (32, 32, 32) after applying maxPooling(2,2) it become = (16, 16, 32)


    #FC layers
    self.fc_layer = nn.Sequential(
        nn.Linear(4 * 4 * 128, 256),
        nn.ReLU(),
        nn.Linear(256, 10)
    )


  # forward_pass
  def forward(self, x):
    x = self.conv_layer(x)
    # to flatten
    x = x.view(x.size(0), -1) #flattening
    x = self.fc_layer(x)
    return x

In [8]:
model = CNN()

In [9]:
#Loss &  optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

###Training the CNN

In [10]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=1.4561131973095867
epoch=2/10 & loss=1.028309328171908
epoch=3/10 & loss=0.8274980123390627
epoch=4/10 & loss=0.6999678906729764
epoch=5/10 & loss=0.6006260793227369
epoch=6/10 & loss=0.5206732851312593
epoch=7/10 & loss=0.43850903506474115
epoch=8/10 & loss=0.3540897892640375
epoch=9/10 & loss=0.2933179180869056
epoch=10/10 & loss=0.23256216899437063


###Evaluation

In [12]:
correct_labels = 0
total_labels = 0
model.eval()
with torch.no_grad():
  for images, labels in testloader:
    outputs = model.forward(images)
    _,predicted = torch.max(outputs, 1)

    correct_labels += (predicted == labels).sum().item()
    total_labels += labels.size(0)

accuracy = correct_labels / total_labels
print(f"Accuracy: {accuracy*100}%")


Accuracy: 94.554%
